In [ ]:
from brainspace.gradient.utils import dominant_set
from brainspace.mesh.mesh_io import read_surface
from brainspace.utils.parcellation import map_to_labels
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from netneurotools import datasets as nntdata
from neuromaps import nulls, images
from scipy.stats import pearsonr, ttest_1samp
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import fdrcorrection
from surfplot import Plot

import bct
import cmasher as cmr
import glob
import matplotlib.pyplot as plt
import natsort
import nibabel as nib
import numpy as np
import os
import pandas as pd
import pickle
import seaborn as sns
import statsmodels.formula.api as smf
import SUITPy.flatmap as flatmap
import warnings
warnings.filterwarnings('ignore')

def plot_subcortex(data, color_range, save_path, surf_file="../utils/atlases/subcorticalSurf.pkl"):
    with open(surf_file, "rb") as f:
        VL, VR = pickle.load(f)

    nROI = len(VL)
    cmap = cmr.get_sub_cmap("coolwarm", 0.01, 0.99)

    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(141, projection="3d")
    ax2 = fig.add_subplot(142, projection="3d")
    ax3 = fig.add_subplot(143, projection="3d")
    ax4 = fig.add_subplot(144, projection="3d")

    vmin, vmax = color_range

    def val_to_color(val):
        if not np.isfinite(val):
            return (0.6, 0.6, 0.6, 1.0)
        frac = (val - vmin) / float(vmax - vmin) if vmax > vmin else 0.5
        frac = max(min(frac, 1.0), 0.0)
        return cmap(frac)

    coords_lists = {ax1: [], ax2: [], ax3: [], ax4: []}

    for p in range(nROI):
        if VL[p]["simplices"].size > 0:
            coords = VL[p]["coords"]
            faces = VL[p]["simplices"]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(data[p + nROI]))
            mesh.set_edgecolor("none")
            ax1.add_collection3d(mesh)
            coords_lists[ax1].append(coords)
    ax1.view_init(elev=0, azim=0)
    ax1.set_box_aspect((1, 1, 1))
    ax1.set_axis_off()

    for p in range(nROI):
        if VL[p]["simplices"].size > 0:
            coords = VL[p]["coords"]
            faces = VL[p]["simplices"]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(data[p + nROI]))
            mesh.set_edgecolor("none")
            ax2.add_collection3d(mesh)
            coords_lists[ax2].append(coords)
    ax2.view_init(elev=0, azim=180)
    ax2.set_box_aspect((1, 1, 1))
    ax2.set_axis_off()

    for p in range(nROI):
        if VR[p]["simplices"].size > 0:
            coords = VR[p]["coords"]
            faces = VR[p]["simplices"]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(data[p]))
            mesh.set_edgecolor("none")
            ax3.add_collection3d(mesh)
            coords_lists[ax3].append(coords)
    ax3.view_init(elev=0, azim=0)
    ax3.set_box_aspect((1, 1, 1))
    ax3.set_axis_off()

    for p in range(nROI):
        if VR[p]["simplices"].size > 0:
            coords = VR[p]["coords"]
            faces = VR[p]["simplices"]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(data[p]))
            mesh.set_edgecolor("none")
            ax4.add_collection3d(mesh)
            coords_lists[ax4].append(coords)
    ax4.view_init(elev=0, azim=180)
    ax4.set_box_aspect((1, 1, 1))
    ax4.set_axis_off()

    for ax in [ax1, ax2, ax3, ax4]:
        if coords_lists[ax]:
            allc = np.concatenate(coords_lists[ax], axis=0)
            ax.auto_scale_xyz(allc[:, 0], allc[:, 1], allc[:, 2])

    plt.tight_layout()
    plt.savefig(save_path, dpi=800, transparent=True)
    plt.close(fig)


def plot_cerebellum(data_array, color_range, save_path):
    label_file = "../utils/atlases/atl-NettekovenAsym32_dseg.label.gii"
    labels = nib.load(label_file)
    label_data = labels.darrays[0].data
    num_vertices = label_data.shape[0]

    data = np.zeros(num_vertices, dtype=float)
    for i, roi_value in enumerate(data_array):
        data[label_data == (i + 1)] = roi_value

    cmap = cmr.get_sub_cmap("coolwarm", 0.01, 0.99)
    flatmap.plot(data, overlay_type="func", cmap=cmap, cscale=color_range, new_figure=True,
                 colorbar=True, bordercolor="w")
    plt.savefig(save_path, dpi=800, transparent=True)
    plt.close()

def get_files(pattern, force_list=False):
    if isinstance(pattern, list):
        pattern = os.path.join(*pattern)
    files = natsort.natsorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError("Pattern could not detect file(s)")
    if (not force_list) & (len(files) == 1):
        return files[0]
    else:
        return files

def _align_labels_to_atlas(x, source_labels, target_labels):
    target = np.unique(target_labels)[1:]
    df1 = pd.DataFrame(target, index=target)
    df2 = pd.DataFrame(x, index=source_labels)
    return pd.concat([df1, df2], axis=1).iloc[:, 1:].values

def weights_to_vertices(data, target, labels=None):
    if isinstance(target, str):
        vertices = nib.load(target).get_fdata().ravel()
    else:
        vertices = target.ravel()
    if labels is not None:
        data = _align_labels_to_atlas(data, labels, vertices)
    mask = vertices != 0
    map_args = dict(target_lab=vertices, mask=mask, fill=np.nan)
    if (len(data.shape) == 1) or (data.shape[1] == 1):
        weights = map_to_labels(data.ravel(), **map_args)
    else:
        weights = [map_to_labels(x, **map_args) for x in data.T]
    return weights

def get_surfaces(style="inflated", load=True):
    surf_path = "../utils/surfaces"
    surfaces = get_files([surf_path, f"*.{style}_*"])
    if load:
        surfs = [read_surface(i) for i in surfaces]
        return dict(zip(["lh", "rh"], surfs))
    else:
        return surfaces

def correlate_rois(wide_df, features=("early_base", "late_early", "late_base")):
    rows = []
    for (roi, roi_ix), g in wide_df.groupby(["roi", "roi_ix"], sort=False):
        for feat in features:
            r, p = pearsonr(g[feat], g["FPC1_z"])
            rows.append({"roi": roi, "roi_ix": roi_ix, "feature": feat, "r": r, "p": p})
    return pd.DataFrame(rows)

def plot_correlations(correlations_data):
    surfaces = get_surfaces()
    for feature in ["early_base", "late_early", "late_base"]:
        fdat = correlations_data[correlations_data["feature"] == feature].copy()
        vmax = 0.43
        cmap = cmr.get_sub_cmap("coolwarm", 0.01, 0.99)

        cortex = fdat[fdat["roi"].astype(str).str.startswith("7Net")].copy()
        cortex_labels = (cortex["roi_ix"].astype(int) - 64).to_list()
        x = weights_to_vertices(cortex["r"].to_numpy(), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", cortex_labels)
        p = Plot(surfaces["lh"], surfaces["rh"])
        p.add_layer(x, cmap=cmap, color_range=(-vmax, vmax), cbar=False)
        _ = p.build(colorbar=False)
        plt.savefig(os.path.join("../figures/network-correlations", f"{feature}_cortex.png"), dpi=800, transparent=True)
        plt.close()

        subcortex = fdat[fdat["roi"].astype(str).str.endswith(("-lh", "-rh"))].sort_values("roi_ix")
        plot_subcortex(data=subcortex["r"].to_numpy(), color_range=(-vmax, vmax), save_path=os.path.join("../figures/network-correlations", f"{feature}_subcortex.png"))

        cerebellum = fdat[fdat["roi"].astype(str).str.endswith(("L", "R"))].sort_values("roi_ix")
        plot_cerebellum(data_array=cerebellum["r"].to_numpy(), color_range=(-vmax, vmax), save_path=os.path.join("../figures/network-correlations", f"{feature}_cerebellum.png"))

def add_network_labels(correlations_df):
    return parse_roi_names(correlations_df)

def parse_roi_names(df, col="roi"):
    roi_cols = ["hemi", "network", "name"]
    out = df.copy()
    out[roi_cols] = out[col].astype(str).str.split("_", n=3, expand=True).iloc[:, 1:]
    return out

def get_schaefer400_parcellation_and_surfaces():
    parcellation_paths = ("../utils/atlases/Schaefer2018_400Parcels_7Networks_32k.L.label.gii",
                          "../utils/atlases/Schaefer2018_400Parcels_7Networks_32k.R.label.gii")
    surface_paths = ("../utils/surfaces/S1200.L.sphere.32k_fs_LR.surf.gii",
                     "../utils/surfaces/S1200.R.sphere.32k_fs_LR.surf.gii")
    return parcellation_paths, surface_paths

def parcel_map_from_correlations(corr_df, feature, network_order=("Vis", "SomMot", "DorsAttn", "Limbic", "Cont", "SalVentAttn", "Default")):
    df = corr_df[corr_df["feature"] == feature].copy()
    df = parse_roi_names(df)
    df["network"] = pd.Categorical(df["network"], categories=list(network_order), ordered=True)

    parcel_map = np.full(400, np.nan, dtype=float)
    ix = df["roi_ix"].to_numpy().astype(int) - 1
    parcel_map[ix] = df["r"].to_numpy()

    network_codes = df["network"].cat.codes.to_numpy()
    network_names = df["network"].cat.categories

    network_labels = np.full(400, np.nan, dtype=float)
    network_labels[ix] = np.where(network_codes >= 0, network_codes.astype(float), np.nan)

    return parcel_map, network_labels, network_names

def spin_null_network_means(parcel_map, network_labels, parcellation_paths, surface_paths, n_perm=1000, seed=1234):
    null_maps = nulls.vasa(parcel_map, parcellation=parcellation_paths, surfaces=surface_paths, n_perm=n_perm, seed=seed)
    if null_maps.shape[0] != n_perm:
        null_maps = null_maps.T

    valid = ~np.isnan(network_labels)
    nets = np.unique(network_labels[valid])

    empirical_means, null_means = {}, {}
    for net in nets:
        mask = network_labels == net
        empirical_means[int(net)] = np.nanmean(parcel_map[mask])
        null_means[int(net)] = np.nanmean(null_maps[:, mask], axis=1)

    return empirical_means, null_means

def spin_pvalues(empirical_means, null_means):
    pvals = {}
    for net, emp in empirical_means.items():
        nullv = null_means[net]
        n = len(nullv)
        p_hi = (1 + np.sum(nullv >= emp)) / (n + 1)
        p_lo = (1 + np.sum(nullv <= emp)) / (n + 1)
        pvals[net] = min(1.0, 2 * min(p_hi, p_lo))
    return pvals

def pval_diff_from_zero(parcel_map, network_labels, net_code):
    mask = network_labels == net_code
    vals = parcel_map[mask]
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return np.nan
    return float(ttest_1samp(vals, popmean=0.0, nan_policy="omit").pvalue)

def plot_spin_boxplot(null_df, emp_df, feature, order, ylim=(-0.25, 0.15)):
    palette = {"Vis": "#6D1D86", "SomMot": "#537EAB", "DorsAttn": "#327321", "SalVentAttn": "#B445F6",
               "Limbic": "#E1F8AE", "Cont": "#CA8E3B", "Default": "#BD4C52"}

    f_null = null_df[null_df["feature"] == feature].copy()
    f_emp  = emp_df[emp_df["feature"] == feature].copy()

    f_null["network_name"] = pd.Categorical(f_null["network_name"], categories=order, ordered=True)
    f_emp["network_name"]  = pd.Categorical(f_emp["network_name"],  categories=order, ordered=True)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax = sns.boxplot(data=f_null, x="network_name", y="null_mean", palette=palette, order=order, fliersize=0, whis=(2.5, 97.5), ax=ax)
    for patch in ax.artists:
        patch.set_alpha(0.5)

    sig_col = "significant_both"
    facecolors = [palette[str(net)] if bool(sig) else "white" for net, sig in zip(f_emp["network_name"], f_emp[sig_col])]

    ax.scatter(f_emp["network_name"], f_emp["empirical_mean"], s=60, c=facecolors, edgecolors="black", linewidths=1, zorder=10)
    ax.axhline(0, linestyle="--", linewidth=1, color="gray")
    ax.set_ylim(*ylim)
    ax.set_title(f"{feature}: spin null vs empirical")
    ax.set_ylabel("Mean correlation")
    ax.set_xlabel("Network")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.savefig(f"../figures/network-correlations/{feature}_boxplot.png", dpi=800, transparent=True)
    plt.close()

def pick_strongest_roi_per_network_feature(corr_net, network_name, feature, use_abs=True):
    df = corr_net[corr_net["roi"].astype(str).str.startswith("7Net")].copy()
    df = df[df["network"] == network_name].copy()
    df = df[df["feature"] == feature].copy()

    df = df.drop_duplicates(subset=["roi", "roi_ix", "network", "feature"]).copy()
    df["score"] = df["r"].abs() if use_abs else df["r"]
    best = df.sort_values("score", ascending=False).iloc[0]

    return {"network": network_name, "feature": feature, "roi": best["roi"], "roi_ix": int(best["roi_ix"]),
            "r": float(best["r"]), "score": float(best["score"])}

def pick_roi_per_network_feature(corr_net, network_name, feature, direction="pos"):
    df = corr_net[corr_net["roi"].astype(str).str.startswith("7Net")].copy()
    df = df[df["network"] == network_name].copy()
    df = df[df["feature"] == feature].copy()

    df = df.drop_duplicates(subset=["roi", "roi_ix", "network", "feature"]).copy()
    if direction == "abs":
        df["score"] = df["r"].abs()
        best = df.sort_values("score", ascending=False).iloc[0]
    elif direction == "pos":
        best = df.sort_values("r", ascending=False).iloc[0]
    elif direction == "neg":
        best = df.sort_values("r", ascending=True).iloc[0]

    return {"network": network_name, "feature": feature, "roi": best["roi"], "roi_ix": int(best["roi_ix"]), "r": float(best["r"])}

def pick_one_limbic_roi_by_rank(corr_net, feat_pos="early_base", feat_neg="late_early", require_signs=True):
    df = corr_net[corr_net["roi"].astype(str).str.startswith("7Net")].copy()
    df = df[df["network"] == "Limbic"].copy()
    df = df[df["feature"].isin([feat_pos, feat_neg])].copy()
    df = df.drop_duplicates(subset=["roi", "roi_ix", "network", "feature"]).copy()

    piv = (
        df.pivot_table(index=["roi", "roi_ix"], columns="feature", values="r", aggfunc="mean")
          .dropna(subset=[feat_pos, feat_neg], how="any")
    )

    if require_signs:
        piv = piv[(piv[feat_pos] > 0) & (piv[feat_neg] < 0)]
        if piv.empty:
            return None

    # ranks with NO ties (critical)
    piv["rank_pos"] = piv[feat_pos].rank(ascending=False, method="first")  # higher early_base is better
    piv["rank_neg"] = piv[feat_neg].rank(ascending=True,  method="first")  # more negative late_early is better

    piv["rank_score"] = piv["rank_pos"] + piv["rank_neg"]

    # deterministic tie-break: prefer better late_early rank, then better early_base rank
    best_idx = (
        piv.sort_values(["rank_score", "rank_neg", "rank_pos"], ascending=[True, True, True])
           .index[0]
    )

    roi, roi_ix = best_idx
    return {
        "network": "Limbic",
        "roi": roi,
        "roi_ix": int(roi_ix),
        f"r_{feat_pos}": float(piv.loc[best_idx, feat_pos]),
        f"r_{feat_neg}": float(piv.loc[best_idx, feat_neg]),
        "rank_pos": int(piv.loc[best_idx, "rank_pos"]),
        "rank_neg": int(piv.loc[best_idx, "rank_neg"]),
        "rank_score": float(piv.loc[best_idx, "rank_score"]),
    }
    
def plot_seed_scatterplots(wide, seeds_df, network_colors, out_dir="../figures/network-correlations"):
    for _, row in seeds_df.iterrows():
        net = row["network"]
        feat = row["feature"]
        roi = row["roi"]
        roi_ix = int(row["roi_ix"])
        df = wide[(wide["roi"] == roi) & (wide["roi_ix"] == roi_ix)].copy()

        fig, ax = plt.subplots(figsize=(3.0, 3.0))
        color = network_colors.get(net, "gray")
        ax.scatter(df[feat], df["FPC1_z"], s=40, c=color, edgecolors="black", linewidths=0.8, alpha=0.9)
        sns.regplot(data=df, x=feat, y="FPC1_z", scatter=False, line_kws={"color": "black", "linewidth": 1.2}, ax=ax)
        ax.set_title(f"{net} | {feat}\nROI {roi_ix} (r={row['r']:.3f})", fontsize=10)
        ax.set_xlabel(feat)
        ax.set_ylabel("FPC1_z")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"seed_{net}_roi_{roi_ix}_{feat}.png"), dpi=800, transparent=True)
        plt.close()

def main():
    os.makedirs("../figures/network-correlations", exist_ok=True)

    gradients = pd.read_csv("../output/subject_gradients.tsv", sep="\t")
    gradients = gradients.groupby(["roi", "roi_ix", "sub", "epoch", "network"], sort=False, as_index=False)["distance"].mean()
    
    learning = pd.read_csv("../output/fpca_scores.csv")
    corr_data = gradients.merge(learning[["subid", "FPC1_z"]], left_on="sub", right_on="subid", how="left")

    wide = (corr_data.pivot_table(index=["roi", "roi_ix", "sub", "FPC1_z"], columns="epoch", values="distance", sort=False).reset_index())
    wide["early_base"] = wide["early"] - wide["base"]
    wide["late_early"] = wide["late"] - wide["early"]
    wide["late_base"]  = wide["late"] - wide["base"]

    correlations_data = correlate_rois(wide, features=["early_base", "late_early", "late_base"])
    plot_correlations(correlations_data)
    corr_net = add_network_labels(correlations_data)
    corr_net.to_csv("../output/roi_correlations_original.csv", index=False)

    cortex_corr = correlations_data[correlations_data["roi"].astype(str).str.startswith("7Net")].copy()
    cortex_corr["roi_ix"] = cortex_corr["roi_ix"] - 64
    parcellation_paths, surface_paths = get_schaefer400_parcellation_and_surfaces()

    all_null, all_emp = [], []
    for feat in ["early_base", "late_early", "late_base"]:
        parcel_map, network_labels, network_names = parcel_map_from_correlations(cortex_corr, feat)

        empirical_means, null_means = spin_null_network_means(parcel_map, network_labels, parcellation_paths=parcellation_paths,
                                      surface_paths=surface_paths, n_perm=1000, seed=1234)
        p_spin = spin_pvalues(empirical_means, null_means)
        p_zero = {net: pval_diff_from_zero(parcel_map, network_labels, net) for net in empirical_means.keys()}

        for net, emp in empirical_means.items():
            net_name = str(network_names[int(net)])
            all_emp.append({"feature": feat, "network_code": int(net), "network_name": net_name, "empirical_mean": emp,
                            "p_spin": p_spin.get(net, np.nan), "p_zero": p_zero.get(net, np.nan)})
            for val in null_means[net]:
                all_null.append({"feature": feat, "network_code": int(net), "network_name": net_name, "null_mean": val})

    null_df = pd.DataFrame(all_null)
    emp_df  = pd.DataFrame(all_emp)

    emp_df["sig_spin"] = emp_df["p_spin"] < 0.05
    emp_df["sig_zero"] = emp_df["p_zero"] < 0.05
    emp_df["significant_both"] = emp_df["sig_spin"] & emp_df["sig_zero"]

    emp_df["sig_spin"] = emp_df["p_spin"] < 0.05
    emp_df["sig_zero"] = emp_df["p_zero"] < 0.05
    emp_df["significant_both"] = emp_df["sig_spin"] & emp_df["sig_zero"]
    
    emp_df.to_csv("../output/spin_test_pvalues_by_network.csv", index=False)

    order = ["Vis", "SomMot", "DorsAttn", "Limbic", "Cont", "SalVentAttn", "Default"]
    for feat in ["early_base", "late_early", "late_base"]:
        plot_spin_boxplot(null_df, emp_df, feat, order=order, ylim=(-0.25, 0.15))

    network_colors = {"Vis": "#6D1D86", "SomMot": "#537EAB", "DorsAttn": "#327321", "SalVentAttn": "#B445F6",
                      "Limbic": "#E1F8AE", "Cont": "#CA8E3B", "Default": "#BD4C52"}

    sig_pairs = emp_df[emp_df["significant_both"]].copy()
    sig_pairs = sig_pairs[sig_pairs["network_name"].isin(["Limbic", "DorsAttn", "Default", "SalVentAttn"])].copy()

    seeds = []
    limbic_feats = sorted(sig_pairs[sig_pairs["network_name"] == "Limbic"]["feature"].unique().tolist())
    if ("early_base" in limbic_feats) and ("late_early" in limbic_feats):
        limbic_pick = pick_one_limbic_roi_by_rank(corr_net, "early_base", "late_early", require_signs=True)
        
        if limbic_pick is not None:
            roi = limbic_pick["roi"]
            roi_ix = limbic_pick["roi_ix"]
            for feat in ["early_base", "late_early"]:
                r_val = float(
                    corr_net[(corr_net["roi"] == roi) & (corr_net["roi_ix"] == roi_ix) & (corr_net["feature"] == feat)]["r"].iloc[0])
                seeds.append({"network": "Limbic", "feature": feat, "roi": roi, "roi_ix": roi_ix, "r": r_val})

    direction_map = {("Limbic", "early_base"): "pos", ("Limbic", "late_early"): "neg", ("Default", "late_early"): "pos",
                     ("SalVentAttn", "late_early"): "neg", ("DorsAttn", "late_base"): "neg"}
    for _, row in sig_pairs.iterrows():
        net = row["network_name"]
        feat = row["feature"]
        if net == "Limbic":
            continue 
        direction = direction_map.get((net, feat), "abs")  
        pick = pick_roi_per_network_feature(corr_net=corr_net, network_name=net, feature=feat, direction=direction)
        if pick is not None:
            seeds.append(pick)

main()